### Kernel: R

## Replicacao dados de cancer de mama utilizando pseudo-valores para S(t)

Formula dos pseudo-valores para S(t):
$$Y_{ij} = n\hat{S}_{KM}(t_j) - (n-1)\hat{S}_{KM}^{(-i)}(t_j)$$

In [1]:
# Bibliotecas utilizadas
library(pacman)
p_load(survival, dplyr, tidyr)


In [2]:
dados_treino <- read.csv("../../../dados/dados-processados/dados_treino.csv", stringsAsFactors = FALSE)
dados_teste <- read.csv("../../../dados/dados-processados/dados_teste.csv", stringsAsFactors = FALSE)

covariaveis_rede <- setdiff(
  intersect(names(dados_treino), names(dados_teste)),
  c("id_paciente", "tempo", "delta_t")
)

prepara_base <- function(df, conjunto_label, covariaveis) {
  base <- df %>%
    select(id_paciente, tempo, delta_t, all_of(covariaveis)) %>%
    mutate(
      id_paciente = as.numeric(id_paciente),
      tempo = as.numeric(tempo),
      delta_t = as.numeric(delta_t),
      conjunto = conjunto_label
    ) %>%
    select(id_paciente, tempo, delta_t, all_of(covariaveis), conjunto) %>%
    filter(tempo > 0, !is.na(delta_t))
  base
}

treino_base <- prepara_base(dados_treino, "treino", covariaveis_rede)
teste_base <- prepara_base(dados_teste, "teste", covariaveis_rede)

cancer_mama <- bind_rows(treino_base, teste_base) %>%
  arrange(id_paciente)

n_distinct(cancer_mama$id_paciente)
head(cancer_mama)

[1] 6076

,id_paciente,tempo,delta_t,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,⋯,rhc_diagnostico_e_tratamentos_anteriores,rhc_idade_no_diagnostico_tumor,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,conjunto
,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<int>,<int>,<chr>,<int>,<int>,<chr>,<int>,<dbl>,<chr>
1,1,16,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,⋯,Com diag./Sem trat.,73,3,Média Renda,5,2014,8500,12,42,treino
2,2,60,1,Não Branca,0,2,Não,Nunca,Sim,SUS,⋯,Sem diag./Sem trat.,75,3,Média Renda,5,2012,8500,-104,69,treino
3,3,1,1,Não Branca,0,2,Não,Nunca,Nunca,SUS,⋯,Sem diag./Sem trat.,93,1,Alta Renda,5,2018,8500,-41,0,teste
4,4,13,1,Não Branca,0,1,Sim,Nunca,Ex-consumidor,SUS,⋯,Com diag./Sem trat.,29,2,Média Renda,1,2013,8500,19,33,teste
5,5,69,1,Não Branca,1,2,Não,Nunca,Nunca,SUS,⋯,Com diag./Sem trat.,38,1,Baixa Renda,1,2010,8500,83,99,treino
6,6,7,1,Não Branca,1,2,Sim,Nunca,Sim,SUS,⋯,Sem diag./Sem trat.,43,1,Alta Renda,2,2017,min100,-10,46,treino


In [3]:
grade_tempos <- quantile(
  treino_base$tempo[treino_base$delta_t > 0],
  probs = seq(0.05, 0.95, length.out = 10),
  names = FALSE
)

grade_tempos

[1]  2.0  7.0 11.0 15.0 20.0 27.0 32.1 41.0 52.9 73.0

In [4]:
# Função para calcular a sobrevivência em tempos específicos

get_sobrevivencia <- function(fit, tempos) {
  summary(fit, times = tempos, extend = TRUE)$surv
}

calcula_pseudo <- function(time, status, tempos) {
  n <- length(time)
  m <- length(tempos)

  fit_completo <- survfit(Surv(time, status) ~ 1)
  s_completo <- get_sobrevivencia(fit_completo, tempos)

  pseudo <- matrix(NA, nrow = n, ncol = m)

  for (i in seq_len(n)) {
    fit_sem_i <- survfit(Surv(time[-i], status[-i]) ~ 1)
    s_sem_i <- get_sobrevivencia(fit_sem_i, tempos)
    pseudo[i, ] <- n * s_completo - (n - 1) * s_sem_i
  }

  colnames(pseudo) <- paste0("t_", round(tempos, 6))
  pseudo
}

# Calcular pseudo para dados de treino

pseudo_treino <- calcula_pseudo(
  time = treino_base$tempo,
  status = treino_base$delta_t,
  tempos = grade_tempos
)

n_train <- nrow(treino_base)
m <- length(grade_tempos)

base_expandida_treino <- treino_base[
  rep(seq_len(n_train), each = m),
  c("id_paciente", covariaveis_rede),
  drop = FALSE
]

entrada_rede_treino <- base_expandida_treino %>%
  mutate(
    tempo = rep(grade_tempos, times = n_train),
    pseudo = as.vector(t(pseudo_treino)),
    conjunto = "treino"
  )

In [5]:
# Não há necessidade de calcular pseudo para teste, pois serão usadas as previsões do modelo para comparação
n_test <- nrow(teste_base)

base_expandida_teste <- teste_base[
  rep(seq_len(n_test), each = m),
  c("id_paciente", covariaveis_rede),
  drop = FALSE
]

entrada_rede_teste <- base_expandida_teste %>%
  mutate(
    tempo = rep(grade_tempos, times = n_test),
    conjunto = "teste",
    pseudo = NA_real_
  )

# Combinar treino e teste
entrada_rede <- bind_rows(entrada_rede_treino, entrada_rede_teste) %>%
  select(id_paciente, all_of(covariaveis_rede), tempo, pseudo, conjunto)

head(entrada_rede, 20)

,id_paciente,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,rhc_estadiamento_clinico,rhc_primeiro_tratamento_recebido_no_hospital_nenhum,⋯,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,tempo,pseudo,conjunto
,<dbl>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<int>,<chr>,<int>,<int>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<chr>
1...1,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,3,Média Renda,5,2014,8500,12,42,2.0,1.00000429,treino
1.1...2,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,3,Média Renda,5,2014,8500,12,42,7.0,1.00011630,treino
1.2...3,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,3,Média Renda,5,2014,8500,12,42,11.0,1.00032896,treino
1.3...4,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,3,Média Renda,5,2014,8500,12,42,15.0,1.00070533,treino
1.4...5,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,3,Média Renda,5,2014,8500,12,42,20.0,-0.03829849,treino
1.5...6,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,3,Média Renda,5,2014,8500,12,42,27.0,-0.03775873,treino
1.6...7,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,3,Média Renda,5,2014,8500,12,42,32.1,-0.03723958,treino
1.7...8,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,3,Média Renda,5,2014,8500,12,42,41.0,-0.03658440,treino
1.8...9,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,3,Média Renda,5,2014,8500,12,42,52.9,-0.03591844,treino


In [6]:
dir_entrada <- file.path("..", "dados", "entrada")
write.csv(entrada_rede, file.path(dir_entrada, "entrada_rede.csv"), row.names = FALSE)

list.files(dir_entrada)

[1] "entrada_rede.csv"